In [0]:
from pyspark.sql.functions import col, when, current_timestamp

In [0]:
src_Table='project_mobility.bronze.taxi_zone_lookup_raw'
target_Table='project_mobility.silver.taxi_zone_enriched'

In [0]:
df = spark.read.table(src_Table)

df_renamed = df.select(
      col("LocationID").cast("int").alias("location_id"),
      col("Borough").alias("borough"),
      col("Zone").alias("zone"),
      col("service_zone").alias("service_zone")
  )

df_derived = df_renamed.withColumns({
    "is_airport":col("zone").isin("JFK Airport", "LaGuardia Airport", "Newark Airport"),
    "is_manhattan": col("borough") == "Manhattan",
    "zone_category": when(col("zone").isin("JFK Airport", "LaGuardia Airport", "Newark Airport"), "Airport")
        .when(col("borough") == "Manhattan", "Manhattan")
        .otherwise("Outer Borough"),
    "ingestion_timestamp": current_timestamp()
  })

df_derived.write.mode("overwrite").saveAsTable(target_Table)
#display(df_derived)